# Lightweight Machine Learning-Based Network Intrusion Detection for Resource-Constrained Edge Devices

## Project Overview

This notebook implements and evaluates a lightweight machine learning-based Network Intrusion Detection System (NIDS) using the Edge-IIoTset dataset.

### Research Objective
To evaluate the trade-off between network attack detection performance and computational resource consumption when using lightweight machine learning approaches.

### Dataset
Edge-IIoTset:
- ML-EdgeIIoT-dataset.csv
- Individual attack and normal traffic CSV files where required.

### Evaluation Environment
- Hardware: MacBook Pro
- Language: Python
- Libraries: Pandas, NumPy, Scikit-learn and Matplotlib.

### Research Scope
The study focuses on controlled host-based evaluation as a proxy for resource-constrained edge deployment. It does not claim physical edge-device validation.

In [3]:
import os
import sys
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: 3.12.5 (v3.12.5:ff3bc82f7c9, Aug  7 2024, 05:32:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform: macOS-13.7.8-x86_64-i386-64bit
Pandas: 3.0.2
NumPy: 1.26.4
Scikit-learn: 1.5.1


## 1.1 Dataset Configuration

The study uses the Edge-IIoTset dataset. The main dataset selected for machine learning experiments is ML-EdgeIIoT-dataset.csv.

The individual attack and normal traffic CSV files will only be examined where necessary for dataset verification and evaluation design.

In [4]:
# Define the dataset location
DATASET_PATH = os.path.join(
    "Edge-IIoTset dataset",
    "Selected dataset for ML and DL",
    "ML-EdgeIIoT-dataset.csv"
)

# Verify that the dataset exists
if os.path.exists(DATASET_PATH):
    print("Dataset found:", DATASET_PATH)
    print("File size:", round(os.path.getsize(DATASET_PATH) / (1024 * 1024), 2), "MB")
else:
    print("Dataset not found. Check the project directory.")

Dataset found: Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv
File size: 78.38 MB


## 2. Dataset Inspection

This section examines the structure of the selected Edge-IIoTset dataset before any preprocessing or model training.

The inspection focuses on:
- Dataset dimensions.
- Feature names and data types.
- Target labels.
- Potential capture or session identifiers.
- Initial sample records.

In [5]:
# Load a small sample for initial inspection
sample_df = pd.read_csv(DATASET_PATH, nrows=5)

# Display dataset structure
print("Number of columns:", len(sample_df.columns))

print("\nColumn names:")
for column in sample_df.columns:
    print("-", column)

print("\nData types:")
display(sample_df.dtypes.to_frame(name="Data Type"))

print("\nFirst five records:")
display(sample_df)

Number of columns: 63

Column names:
- frame.time
- ip.src_host
- ip.dst_host
- arp.dst.proto_ipv4
- arp.opcode
- arp.hw.size
- arp.src.proto_ipv4
- icmp.checksum
- icmp.seq_le
- icmp.transmit_timestamp
- icmp.unused
- http.file_data
- http.content_length
- http.request.uri.query
- http.request.method
- http.referer
- http.request.full_uri
- http.request.version
- http.response
- http.tls_port
- tcp.ack
- tcp.ack_raw
- tcp.checksum
- tcp.connection.fin
- tcp.connection.rst
- tcp.connection.syn
- tcp.connection.synack
- tcp.dstport
- tcp.flags
- tcp.flags.ack
- tcp.len
- tcp.options
- tcp.payload
- tcp.seq
- tcp.srcport
- udp.port
- udp.stream
- udp.time_delta
- dns.qry.name
- dns.qry.name.len
- dns.qry.qu
- dns.qry.type
- dns.retransmission
- dns.retransmit_request
- dns.retransmit_request_in
- mqtt.conack.flags
- mqtt.conflag.cleansess
- mqtt.conflags
- mqtt.hdrflags
- mqtt.len
- mqtt.msg_decoded_as
- mqtt.msg
- mqtt.msgtype
- mqtt.proto_len
- mqtt.protoname
- mqtt.topic
- mqtt.topic_

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64



First five records:


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [6]:
# Count records using the CSV file structure
with open(DATASET_PATH, "r", encoding="utf-8") as file:
    total_rows = sum(1 for _ in file) - 1

print("Total records:", f"{total_rows:,}")
print("Total features:", len(sample_df.columns))

Total records: 157,800
Total features: 63


## 3. Target Variable and Data Type Inspection

This section examines the target variables, `Attack_label` and `Attack_type`, to understand how normal and malicious traffic is represented. It also checks the data types of the dataset fields before preprocessing.

In [7]:
# Inspect data types
display(sample_df.dtypes.to_frame(name="Data Type"))

# Load only the target columns
target_df = pd.read_csv(
    DATASET_PATH,
    usecols=["Attack_label", "Attack_type"]
)

# Examine target values
print("Attack_label distribution:")
display(target_df["Attack_label"].value_counts(dropna=False))

print("\nAttack_type distribution:")
display(target_df["Attack_type"].value_counts(dropna=False))

# Check missing target values
print("\nMissing target values:")
display(target_df[["Attack_label", "Attack_type"]].isnull().sum())

,Data Type
frame.time,float64
ip.src_host,str
ip.dst_host,float64
arp.dst.proto_ipv4,float64
arp.opcode,float64
...,...
mbtcp.len,float64
mbtcp.trans_id,float64
mbtcp.unit_id,float64
Attack_label,int64


Attack_label distribution:


Attack_label
1    133499
0     24301
Name: count, dtype: int64


Attack_type distribution:


Attack_type
Normal                   24301
DDoS_UDP                 14498
DDoS_ICMP                14090
Ransomware               10925
DDoS_HTTP                10561
SQL_injection            10311
Uploading                10269
DDoS_TCP                 10247
Backdoor                 10195
Vulnerability_scanner    10076
Port_Scanning            10071
XSS                      10052
Password                  9989
MITM                      1214
Fingerprinting            1001
Name: count, dtype: int64


Missing target values:


Attack_label    0
Attack_type     0
dtype: int64

In [8]:
# Load the two target columns
target_df = pd.read_csv(
    DATASET_PATH,
    usecols=["Attack_label", "Attack_type"]
)

print("Attack_label distribution:")
display(
    target_df["Attack_label"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("Attack_type distribution:")
display(
    target_df["Attack_type"]
    .value_counts(dropna=False)
    .to_frame("Count")
)

print("Missing target values:")
display(target_df.isnull().sum())

Attack_label distribution:


,Count
Attack_label,
1,133499
0,24301


Attack_type distribution:


,Count
Attack_type,
Normal,24301
DDoS_UDP,14498
DDoS_ICMP,14090
Ransomware,10925
DDoS_HTTP,10561
SQL_injection,10311
Uploading,10269
DDoS_TCP,10247
Backdoor,10195


Missing target values:


Attack_label    0
Attack_type     0
dtype: int64

## 4. Data Quality Assessment

This section examines missing values, duplicate records and the general quality of the dataset. The results will help identify data preparation requirements before model development.

In [9]:
# Load the approved dataset
df = pd.read_csv(DATASET_PATH)

# Dataset dimensions
print("Dataset shape:", df.shape)

# Missing values
missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": (missing_values / len(df)) * 100
})

display(
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing Values", ascending=False)
)

# Duplicate records
duplicate_count = df.duplicated().sum()

print("Duplicate records:", f"{duplicate_count:,}")
print(
    "Duplicate percentage:",
    f"{(duplicate_count / len(df)) * 100:.2f}%"
)

/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1121213083.py:2: DtypeWarning: Columns (0: arp.dst.proto_ipv4, 1: arp.src.proto_ipv4, 2: http.file_data, 3: http.request.uri.query, 4: http.request.method, 5: http.referer, 6: http.request.full_uri, 7: http.request.version, 8: tcp.options, 9: tcp.payload, 10: tcp.srcport, 11: dns.qry.name.len, 12: mqtt.conack.flags, 13: mqtt.msg, 14: mqtt.protoname, 15: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH)


Dataset shape: (157800, 63)


,Missing Values,Missing Percentage (%)


Duplicate records: 814
Duplicate percentage: 0.52%


In [10]:
# Confirm missing values across all columns
print("Total missing values:", df.isnull().sum().sum())

# Show columns containing missing values
missing_columns = df.isnull().sum()
display(missing_columns[missing_columns > 0])

# Inspect duplicate records
duplicate_rows = df[df.duplicated(keep=False)]

print("Rows involved in duplication:", len(duplicate_rows))
display(duplicate_rows.head(10))

Total missing values: 0


Series([], dtype: int64)

Rows involved in duplication: 821


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
5,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
6,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
7,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
8,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
9,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


## 5. Feature and Capture Identifier Assessment

This section examines the available network traffic fields to identify potential identifiers that could support capture/session-grouped evaluation. It also distinguishes network metadata from potential predictive features.

In [11]:
# Display all columns with their data types
feature_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique(dropna=False).values
})

display(feature_info)

# Inspect columns that may contain capture or session information
identifier_keywords = [
    "capture", "session", "flow", "stream",
    "file", "time", "id"
]

candidate_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in identifier_keywords)
]

print("Potential identifier columns:")
for col in candidate_columns:
    print(f"{col}: {df[col].nunique(dropna=False):,} unique values")

,Column,Data Type,Unique Values
0,frame.time,str,155186
1,ip.src_host,str,19090
2,ip.dst_host,str,8084
3,arp.dst.proto_ipv4,object,9
4,arp.opcode,float64,3
...,...,...,...
58,mbtcp.len,float64,1
59,mbtcp.trans_id,float64,1
60,mbtcp.unit_id,float64,1
61,Attack_label,int64,2


Potential identifier columns:
frame.time: 155,186 unique values
icmp.transmit_timestamp: 84 unique values
http.file_data: 497 unique values
udp.stream: 14,492 unique values
udp.time_delta: 39 unique values
mbtcp.trans_id: 1 unique values
mbtcp.unit_id: 1 unique values


In [12]:
from pathlib import Path

# Locate the approved dataset directory
dataset_root = Path("Edge-IIoTset dataset")

print("Dataset files available for capture-level investigation:\n")

for file in sorted(dataset_root.rglob("*.csv")):
    if file.name == "ML-EdgeIIoT-dataset.csv":
        continue

    print(f"{file.name} | {file.stat().st_size / (1024**2):.2f} MB")

Dataset files available for capture-level investigation:

Backdoor_attack.csv | 17.06 MB
DDoS_HTTP_Flood_attack.csv | 91.28 MB
DDoS_ICMP_Flood_attack.csv | 850.75 MB
DDoS_TCP_SYN_Flood_attack.csv | 897.59 MB
DDoS_UDP_Flood_attack.csv | 858.53 MB
MITM_attack.csv | 0.31 MB
OS_Fingerprinting_attack.csv | 0.30 MB
Password_attack.csv | 623.03 MB
Port_Scanning_attack.csv | 7.00 MB
Ransomware_attack.csv | 7.24 MB
SQL_injection_attack.csv | 25.69 MB
Uploading_attack.csv | 16.70 MB
Vulnerability_scanner_attack.csv | 167.61 MB
XSS_attack.csv | 6.64 MB
Distance.csv | 349.10 MB
Flame_Sensor.csv | 327.61 MB
Heart_Rate.csv | 50.13 MB
IR_Receiver.csv | 393.62 MB
Modbus.csv | 64.75 MB
Soil_Moisture.csv | 363.91 MB
Sound_Sensor.csv | 456.78 MB
Temperature_and_Humidity.csv | 879.85 MB
Water_Level.csv | 697.57 MB
phValue.csv | 226.33 MB
DNN-EdgeIIoT-dataset.csv | 1161.02 MB


In [13]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

files_to_inspect = [
    "MITM_attack.csv",
    "OS_Fingerprinting_attack.csv",
    "Backdoor_attack.csv",
    "Port_Scanning_attack.csv",
    "Distance.csv",
    "Heart_Rate.csv"
]

for filename in files_to_inspect:
    file_path = next(dataset_root.rglob(filename))

    print(f"\n{'=' * 60}")
    print("FILE:", filename)

    sample = pd.read_csv(file_path, nrows=5)

    print("Columns:", len(sample.columns))
    print("Column names:")
    print(sample.columns.tolist())

    print("\nFirst five records:")
    display(sample.head())


FILE: MITM_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgt

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM



FILE: OS_Fingerprinting_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 22:14:30.939803000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x00002ea2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
1,2021 22:14:30.939896000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0x000036a2,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
2,2021 22:14:31.371846000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x0000058a,256.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
3,2021 22:14:31.371927000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0x00000d8a,256.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting
4,2021 22:14:31.910480000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0x0000d3ea,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,OS_Fingerprinting



FILE: Backdoor_attack.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 17:58:23.150209000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
1,2021 17:58:23.150231000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
2,2021 17:58:23.166306000,192.168.0.170,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
3,2021 17:58:23.269052000,0,0,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor
4,2021 17:58:23.659941000,192.168.0.128,192.168.0.170,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,Backdoor


... (2 outputs left out)


FILE: Distance.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgtype

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 23:58:21.314757000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 23:58:21.314960000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
2,2021 23:58:21.315192000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
3,2021 23:58:21.315258000,192.168.1.128,192.168.1.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 23:58:21.315294000,192.168.1.101,192.168.1.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal



FILE: Heart_Rate.csv
Columns: 63
Column names:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.file_data', 'http.content_length', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.options', 'tcp.payload', 'tcp.seq', 'tcp.srcport', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.name.len', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msg', 'mqtt.msgty

,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 02:39:39.986982000,192.168.3.18,192.168.3.12,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 02:39:39.987068000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
2,2021 02:39:40.000498000,192.168.3.18,192.168.3.12,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
3,2021 02:39:40.000575000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 02:39:40.001196000,192.168.3.12,192.168.3.18,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal


In [14]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

capture_files = [
    file for file in dataset_root.rglob("*.csv")
    if file.name != "ML-EdgeIIoT-dataset.csv"
    and file.name != "DNN-EdgeIIoT-dataset.csv"
]

results = []

for file in sorted(capture_files):

    # Read only the target columns
    sample = pd.read_csv(
        file,
        usecols=["Attack_label", "Attack_type"]
    )

    results.append({
        "File": file.name,
        "Records": len(sample),
        "Attack Types": ", ".join(
            sample["Attack_type"].dropna().unique().astype(str)
        ),
        "Labels": ", ".join(
            sample["Attack_label"].dropna().unique().astype(str)
        )
    })

capture_summary = pd.DataFrame(results)

display(capture_summary)

,File,Records,Attack Types,Labels
0,Backdoor_attack.csv,24862,Backdoor,1
1,DDoS_HTTP_Flood_attack.csv,229022,DDoS_HTTP,1
2,DDoS_ICMP_Flood_attack.csv,2914354,DDoS_ICMP,1
3,DDoS_TCP_SYN_Flood_attack.csv,2020120,DDoS_TCP,1
4,DDoS_UDP_Flood_attack.csv,3201626,DDoS_UDP,1
5,MITM_attack.csv,1229,MITM,1
6,OS_Fingerprinting_attack.csv,1001,OS_Fingerprinting,1
7,Password_attack.csv,1053385,Password,1
8,Port_Scanning_attack.csv,22564,Port_Scanning,1
9,Ransomware_attack.csv,10925,Ransomware,1


In [15]:
# Compare selected ML dataset counts with full capture-file counts

capture_counts = capture_summary[
    ["File", "Records", "Attack Types"]
].copy()

# Standardise the one known naming difference
capture_counts["Attack Types"] = capture_counts["Attack Types"].replace(
    {"OS_Fingerprinting": "Fingerprinting"}
)

capture_counts = (
    capture_counts.groupby("Attack Types")["Records"]
    .sum()
    .rename("Capture File Records")
)

ml_counts = (
    df["Attack_type"]
    .value_counts()
    .rename("Selected ML Records")
)

comparison = pd.concat(
    [capture_counts, ml_counts],
    axis=1
).fillna(0)

comparison["Selected ML Records"] = (
    comparison["Selected ML Records"].astype(int)
)

comparison["Capture File Records"] = (
    comparison["Capture File Records"].astype(int)
)

comparison["ML Dataset Coverage (%)"] = (
    comparison["Selected ML Records"]
    / comparison["Capture File Records"] * 100
).round(2)

display(comparison.sort_values("Selected ML Records", ascending=False))

,Capture File Records,Selected ML Records,ML Dataset Coverage (%)
Normal,11209913,24301,0.22
DDoS_UDP,3201626,14498,0.45
DDoS_ICMP,2914354,14090,0.48
Ransomware,10925,10925,100.00
DDoS_HTTP,229022,10561,4.61
SQL_injection,51203,10311,20.14
Uploading,37634,10269,27.29
DDoS_TCP,2020120,10247,0.51
Backdoor,24862,10195,41.01
Vulnerability_scanner,145869,10076,6.91


In [16]:
from pathlib import Path
import pandas as pd

dataset_root = Path("Edge-IIoTset dataset")

# Use several fields to identify sampled records
check_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "Attack_label",
    "Attack_type"
]

# Map selected dataset labels to their capture filenames
capture_mapping = {
    "Backdoor": "Backdoor_attack.csv",
    "DDoS_HTTP": "DDoS_HTTP_Flood_attack.csv",
    "DDoS_ICMP": "DDoS_ICMP_Flood_attack.csv",
    "DDoS_TCP": "DDoS_TCP_SYN_Flood_attack.csv",
    "DDoS_UDP": "DDoS_UDP_Flood_attack.csv",
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv",
    "Password": "Password_attack.csv",
    "Port_Scanning": "Port_Scanning_attack.csv",
    "Ransomware": "Ransomware_attack.csv",
    "SQL_injection": "SQL_injection_attack.csv",
    "Uploading": "Uploading_attack.csv",
    "Vulnerability_scanner": "Vulnerability_scanner_attack.csv",
    "XSS": "XSS_attack.csv"
}

normal_files = [
    "Distance.csv", "Flame_Sensor.csv", "Heart_Rate.csv",
    "IR_Receiver.csv", "Modbus.csv", "Soil_Moisture.csv",
    "Sound_Sensor.csv", "Temperature_and_Humidity.csv",
    "Water_Level.csv", "phValue.csv"
]

# Select up to five records per category
samples = {}

for category in df["Attack_type"].unique():
    samples[category] = df.loc[
        df["Attack_type"] == category, check_columns
    ].sample(
        n=min(5, (df["Attack_type"] == category).sum()),
        random_state=42
    )

# Convert values to comparable strings
def make_keys(data):
    return set(
        pd.util.hash_pandas_object(
            data[check_columns].astype("string").fillna("<NA>"),
            index=False
        ).tolist()
    )

results = []

for category, sample in samples.items():

    if category == "Normal":
        files_to_check = normal_files
    else:
        files_to_check = [capture_mapping[category]]

    sample_keys = make_keys(sample)
    matched_keys = set()

    for filename in files_to_check:
        file_path = next(dataset_root.rglob(filename))

        for chunk in pd.read_csv(
            file_path,
            usecols=check_columns,
            chunksize=100_000
        ):
            matched_keys.update(
                make_keys(chunk).intersection(sample_keys)
            )

    results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched Records": len(matched_keys)
    })

match_results = pd.DataFrame(results)
display(match_results)

/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns (0: ip.dst_host) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/var/folders/n9/xrsn60p14h9cv9mjt3n_9gf00000gp/T/ipykernel_17633/1398151148.py:77: DtypeWarning: Columns

,Attack Type,Sample Records,Matched Records
0,MITM,5,0
1,Fingerprinting,5,0
2,Ransomware,5,5
3,Uploading,5,5
4,SQL_injection,5,5
5,DDoS_HTTP,5,5
6,DDoS_TCP,5,5
7,Password,5,5
8,Port_Scanning,5,5
9,Vulnerability_scanner,5,5


In [20]:
def make_keys(data, columns):
    return set(
        pd.util.hash_pandas_object(
            data[columns].astype("string").fillna("<NA>"),
            index=False
        ).tolist()
    )

In [21]:
diagnostic_columns = [
    col for col in df.columns
    if col not in ["frame.time", "Attack_type"]
]

diagnostic_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

diagnostic_results = []

for category, filename in diagnostic_mapping.items():

    sample = df.loc[
        df["Attack_type"] == category,
        diagnostic_columns
    ].sample(n=5, random_state=42)

    sample_keys = make_keys(sample, diagnostic_columns)

    matched_keys = set()
    file_path = next(dataset_root.rglob(filename))

    for chunk in pd.read_csv(
        file_path,
        usecols=diagnostic_columns,
        chunksize=100_000
    ):
        chunk_keys = make_keys(chunk, diagnostic_columns)
        matched_keys.update(chunk_keys.intersection(sample_keys))

    diagnostic_results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched Records": len(matched_keys)
    })

display(pd.DataFrame(diagnostic_results))

,Attack Type,Sample Records,Matched Records
0,MITM,5,0
1,Fingerprinting,5,0


In [23]:
# Inspect selected ML records against their original capture files

inspection_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "udp.stream",
    "Attack_label",
    "Attack_type"
]

inspection_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

for category, filename in inspection_mapping.items():

    print(f"\n{'='*60}")
    print(f"ATTACK TYPE: {category}")
    print(f"{'='*60}")

    selected_sample = df.loc[
        df["Attack_type"] == category,
        inspection_columns
    ].sample(n=3, random_state=42)

    capture_path = next(dataset_root.rglob(filename))

    capture_data = pd.read_csv(
        capture_path,
        usecols=inspection_columns
    )

    print("\nSELECTED ML DATASET — SAMPLE:")
    display(selected_sample)

    print("\nCAPTURE FILE — FIRST THREE RECORDS:")
    display(capture_data[inspection_columns].head(3))

    print("\nDATA TYPES — SELECTED DATASET:")
    print(df[inspection_columns].dtypes)

    print("\nDATA TYPES — CAPTURE FILE:")
    print(capture_data.dtypes)


ATTACK TYPE: MITM

SELECTED ML DATASET — SAMPLE:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
739,0.0,0,0.0,DESKTOP-UHF0SF2.local,0.0,0.0,1,MITM
788,0.0,0,0.0,0,0.0,0.0,1,MITM
43,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,MITM



CAPTURE FILE — FIRST THREE RECORDS:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
0,0.0,0,0.0,_ipps._tcp.local,0,16.0,1,MITM
1,6.0,192.168.0.152,0.0,0,0,0.0,1,MITM
2,6.0,192.168.0.101,0.0,0,0,0.0,1,MITM



DATA TYPES — SELECTED DATASET:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.srcport      object
tcp.dstport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

DATA TYPES — CAPTURE FILE:
frame.time      float64
ip.src_host         str
ip.dst_host     float64
tcp.dstport         str
tcp.srcport         str
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

ATTACK TYPE: Fingerprinting

SELECTED ML DATASET — SAMPLE:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
1735,2021 22:22:32.191112000,192.168.0.128,192.168.0.170,65535.0,4683.0,0.0,1,Fingerprinting
2155,2021 22:27:37.774518000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,Fingerprinting
1955,2021 22:25:20.622133000,192.168.0.128,192.168.0.170,65535.0,43567.0,0.0,1,Fingerprinting



CAPTURE FILE — FIRST THREE RECORDS:


,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Attack_type
0,2021 22:14:30.939803000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,OS_Fingerprinting
1,2021 22:14:30.939896000,192.168.0.128,192.168.0.170,0.0,0.0,0.0,1,OS_Fingerprinting
2,2021 22:14:31.371846000,192.168.0.170,192.168.0.128,0.0,0.0,0.0,1,OS_Fingerprinting



DATA TYPES — SELECTED DATASET:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.srcport      object
tcp.dstport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object

DATA TYPES — CAPTURE FILE:
frame.time          str
ip.src_host         str
ip.dst_host         str
tcp.dstport     float64
tcp.srcport     float64
udp.stream      float64
Attack_label      int64
Attack_type         str
dtype: object


In [24]:
from decimal import Decimal, InvalidOperation

def normalise_value(value):
    if pd.isna(value):
        return "<NA>"

    value = str(value).strip()

    try:
        number = Decimal(value)
        if number.is_finite():
            return str(number.normalize())
    except InvalidOperation:
        pass

    return value


def normalised_keys(data, columns):
    normalised = data[columns].apply(
        lambda column: column.map(normalise_value)
    )

    return set(
        pd.util.hash_pandas_object(
            normalised,
            index=False
        )
    )


comparison_columns = [
    "frame.time",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport",
    "udp.stream",
    "Attack_label"
]

diagnostic_mapping = {
    "MITM": "MITM_attack.csv",
    "Fingerprinting": "OS_Fingerprinting_attack.csv"
}

normalised_results = []

for category, filename in diagnostic_mapping.items():

    sample = df.loc[
        df["Attack_type"] == category,
        comparison_columns
    ].sample(n=5, random_state=42)

    sample_keys = normalised_keys(sample, comparison_columns)

    capture_path = next(dataset_root.rglob(filename))
    matched_keys = set()

    for chunk in pd.read_csv(
        capture_path,
        usecols=comparison_columns,
        chunksize=100_000
    ):
        chunk_keys = normalised_keys(chunk, comparison_columns)
        matched_keys.update(chunk_keys.intersection(sample_keys))

    normalised_results.append({
        "Attack Type": category,
        "Sample Records": len(sample),
        "Matched After Normalisation": len(matched_keys)
    })

display(pd.DataFrame(normalised_results))

,Attack Type,Sample Records,Matched After Normalisation
0,MITM,5,4
1,Fingerprinting,5,5


In [25]:
# Identify unmatched MITM sample records

category = "MITM"
filename = "MITM_attack.csv"

sample = df.loc[
    df["Attack_type"] == category,
    comparison_columns
].sample(n=5, random_state=42).copy()

def normalised_tuples(data, columns):
    normalised = data[columns].apply(
        lambda column: column.map(normalise_value)
    )
    return set(map(tuple, normalised.to_numpy()))

sample_tuples = normalised_tuples(sample, comparison_columns)

capture_path = next(dataset_root.rglob(filename))
capture_tuples = set()

for chunk in pd.read_csv(
    capture_path,
    usecols=comparison_columns,
    chunksize=100_000
):
    capture_tuples.update(
        normalised_tuples(chunk, comparison_columns)
    )

unmatched_mask = [
    row not in capture_tuples
    for row in sample_tuples
]

# Preserve the original sample row order
sample["Match Status"] = [
    "Unmatched" if tuple(
        normalise_value(value) for value in row
    ) not in capture_tuples else "Matched"
    for row in sample[comparison_columns].to_numpy()
]

display(sample)

,frame.time,ip.src_host,ip.dst_host,tcp.srcport,tcp.dstport,udp.stream,Attack_label,Match Status
739,0.0,0,0.0,DESKTOP-UHF0SF2.local,0.0,0.0,1,Matched
788,0.0,0,0.0,0,0.0,0.0,1,Matched
43,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,Matched
155,0.0,0,0.0,DESKTOP-UHF0SF2,0.0,0.0,1,Matched
494,0.0,0,0.0,_googlecast._tcp.local,0.0,0.0,1,Matched


In [26]:
# Final assessment of candidate grouping fields

group_candidates = [
    "udp.stream",
    "ip.src_host",
    "ip.dst_host",
    "tcp.srcport",
    "tcp.dstport"
]

group_assessment = []

for column in group_candidates:
    group_assessment.append({
        "Feature": column,
        "Data Type": str(df[column].dtype),
        "Unique Values": df[column].nunique(),
        "Missing Values": df[column].isna().sum(),
        "Unique Value Ratio (%)": round(
            df[column].nunique() / len(df) * 100, 2
        )
    })

display(pd.DataFrame(group_assessment))

,Feature,Data Type,Unique Values,Missing Values,Unique Value Ratio (%)
0,udp.stream,float64,14492,0,9.18
1,ip.src_host,str,19090,0,12.10
2,ip.dst_host,str,8084,0,5.12
3,tcp.srcport,object,33551,0,21.26
4,tcp.dstport,float64,23188,0,14.69


In [27]:
# Section 3: Initial feature preparation

# Preserve the original dataframe
model_df = df.copy()

# Define classification targets
binary_target = "Attack_label"
multiclass_target = "Attack_type"

# Separate input features from target variables
X = model_df.drop(
    columns=[binary_target, multiclass_target]
)

y_binary = model_df[binary_target]
y_multiclass = model_df[multiclass_target]

# Identify numerical and categorical features
numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Dataset shape:", X.shape)
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nBinary target shape:", y_binary.shape)
print("Multiclass target shape:", y_multiclass.shape)

Dataset shape: (157800, 61)
Numerical features: 42
Categorical features: 19

Numerical features:
['arp.opcode', 'arp.hw.size', 'icmp.checksum', 'icmp.seq_le', 'icmp.transmit_timestamp', 'icmp.unused', 'http.content_length', 'http.response', 'http.tls_port', 'tcp.ack', 'tcp.ack_raw', 'tcp.checksum', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.dstport', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'tcp.seq', 'udp.port', 'udp.stream', 'udp.time_delta', 'dns.qry.name', 'dns.qry.qu', 'dns.qry.type', 'dns.retransmission', 'dns.retransmit_request', 'dns.retransmit_request_in', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg_decoded_as', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']

Categorical features:
['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.reques